In [1]:
from pyspark.sql import functions as f
from pyspark.sql.functions import *
from datetime import datetime

In [2]:
bi_carteira_cobranca = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_CARTEIRA_COBRANCA/')

19799 [Thread-5] WARN  org.apache.hadoop.metrics2.impl.MetricsConfig  - Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [3]:
bi_contrato = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_CONTRATO/')

base_zeus = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BASE_ZEUS/')

titulos_vertia = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_TITULOS_VERTIA/')

bi_originais = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_ORIGINAIS/')

bi_acordos_promessas = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_ACORDOS_PROMESSAS/')

polos_unidades = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/POLOS_PARCEIROS_E_PROPRIOS/')

bi_recebimento = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_RECEBIMENTOS/')

In [76]:
# 1) validacao qtd
bi_recebimento.count()

173376

In [5]:
# 2) validacao qtd
bi_recebimento.filter(f.col("STA_TITULO") == "PAGO").count()

173376

In [77]:
def join_recebimento(bi_recebimento, bi_acordos_promessas):
    bi_acordos_promessas = bi_acordos_promessas.filter(f.col("NUM_QTDE_PARCELAS") == 1 )
    
    df_bi_recebimento = (bi_recebimento.join(bi_acordos_promessas, 
                           bi_recebimento['cod_parcelamento'] == bi_acordos_promessas['cod_parcelamento'], "left"))
    return df_bi_recebimento.drop(bi_acordos_promessas['cod_parcelamento']).drop(bi_acordos_promessas['COD_CONTRATO'])

In [78]:
def join_bi_contrato(df_bi_recebimento, bi_contrato):
    df_bi_recebimento = (df_bi_recebimento.join(bi_contrato, 
                           df_bi_recebimento['COD_CONTRATO'] == bi_contrato['COD_CONTRATO'], "left"))
    return df_bi_recebimento.drop(bi_contrato['COD_CONTRATO'])

In [79]:
## filtrei somente os campos que preciso na CARTEIRA
def join_carteira_cobranca(df_bi_recebimento, bi_carteira_cobranca):
    bi_carteira_cobranca = bi_carteira_cobranca.select(f.col('COD_CONTRATO'),f.col('NOM_AGENCIA_EXTERNA'),f.col('TIP_MATRICULA'))
    df_bi_recebimento = (df_bi_recebimento.join(bi_carteira_cobranca, 
                           df_bi_recebimento['COD_CONTRATO'] == bi_carteira_cobranca['COD_CONTRATO'], "left"))
    return df_bi_recebimento.drop(bi_carteira_cobranca['COD_CONTRATO']).drop(bi_carteira_cobranca['TIP_MATRICULA'])

In [80]:
## nao precisa pegar o numero ele ja foi pego na CONTRATOS
def join_base_zeus(df_bi_recebimento, base_zeus):
    ##df_bi_recebimento =  (df_bi_recebimento.withColumn("COD_CPFCGC_PES", f.when(f.col("TIP_MATRICULA") == "PRESENCIAL",
     ##                   f.concat(
     ##                   f.lpad(f.col("COD_FILIAL"),3,'0'), f.lit(""), f.substring(f.col('COD_CONTRATO_ORIGEM'), 2, 15)))
      ##                   .otherwise(f.concat(f.lit("EAD"),f.lit(""), f.col("COD_CONTRATO_ORIGEM")))))
    df_bi_recebimento = (df_bi_recebimento.join(base_zeus, 
                       df_bi_recebimento['COD_CPFCGC_PES'] == base_zeus['COD_RA_ZEUS'], "left"))
    return df_bi_recebimento.drop(base_zeus['COD_RA_ZEUS'])  

In [ ]:
df = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/ALUNSITU_EAD/')

In [113]:
df = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/STG_ACORDOS/')

ERROR:py4j.java_gateway:An error occurred while trying to connect to the Java server (127.0.0.1:43401)
Traceback (most recent call last):
  File "/pyspark-library/.venv/lib/python3.7/site-packages/pyspark/python/lib/py4j-0.10.9-src.zip/py4j/java_gateway.py", line 977, in _get_connection
    connection = self.deque.pop()
IndexError: pop from an empty deque

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/pyspark-library/.venv/lib/python3.7/site-packages/pyspark/python/lib/py4j-0.10.9-src.zip/py4j/java_gateway.py", line 1115, in start
    self.socket.connect((self.address, self.port))
ConnectionRefusedError: [Errno 111] Connection refused


Py4JNetworkError: An error occurred while trying to connect to the Java server (127.0.0.1:43401)

In [114]:
from pyspark.sql import functions as f
from pyspark.sql.functions import *
from datetime import datetime

In [116]:
stg_acordos = spark.read.parquet ('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-business/stg_acordos/')

ERROR:py4j.java_gateway:An error occurred while trying to connect to the Java server (127.0.0.1:43401)
Traceback (most recent call last):
  File "/pyspark-library/.venv/lib/python3.7/site-packages/pyspark/python/lib/py4j-0.10.9-src.zip/py4j/java_gateway.py", line 977, in _get_connection
    connection = self.deque.pop()
IndexError: pop from an empty deque

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/pyspark-library/.venv/lib/python3.7/site-packages/pyspark/python/lib/py4j-0.10.9-src.zip/py4j/java_gateway.py", line 1115, in start
    self.socket.connect((self.address, self.port))
ConnectionRefusedError: [Errno 111] Connection refused


Py4JNetworkError: An error occurred while trying to connect to the Java server (127.0.0.1:43401)

In [115]:
df = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-business/stg_acordos/part-00000-c0e1e313-c55e-4f26-8ecc-bff0a277a116-c000.snappy.parquet')

ERROR:py4j.java_gateway:An error occurred while trying to connect to the Java server (127.0.0.1:43401)
Traceback (most recent call last):
  File "/pyspark-library/.venv/lib/python3.7/site-packages/pyspark/python/lib/py4j-0.10.9-src.zip/py4j/java_gateway.py", line 977, in _get_connection
    connection = self.deque.pop()
IndexError: pop from an empty deque

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/pyspark-library/.venv/lib/python3.7/site-packages/pyspark/python/lib/py4j-0.10.9-src.zip/py4j/java_gateway.py", line 1115, in start
    self.socket.connect((self.address, self.port))
ConnectionRefusedError: [Errno 111] Connection refused


Py4JNetworkError: An error occurred while trying to connect to the Java server (127.0.0.1:43401)

In [104]:
df_alunsitu_ead = (df.filter(~(f.col('DAT_STCO_ACDM')
                     .isin('18/12/0015','23/09/0020','10/12/0200','01/12/0206','05/12/0206','10/12/0206','14/02/0208','17/02/0208','02/09/0218','12/12/0220'))))

In [111]:
stg_acordos = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-business/stg_acordos/')

ERROR:py4j.java_gateway:An error occurred while trying to connect to the Java server (127.0.0.1:43401)
Traceback (most recent call last):
  File "/pyspark-library/.venv/lib/python3.7/site-packages/pyspark/python/lib/py4j-0.10.9-src.zip/py4j/java_gateway.py", line 977, in _get_connection
    connection = self.deque.pop()
IndexError: pop from an empty deque

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/pyspark-library/.venv/lib/python3.7/site-packages/pyspark/python/lib/py4j-0.10.9-src.zip/py4j/java_gateway.py", line 1115, in start
    self.socket.connect((self.address, self.port))
ConnectionRefusedError: [Errno 111] Connection refused


Py4JNetworkError: An error occurred while trying to connect to the Java server (127.0.0.1:43401)

In [110]:
stg_acordos = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-business/stg_acordos/')

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/pyspark-library/.venv/lib/python3.7/site-packages/pyspark/python/lib/py4j-0.10.9-src.zip/py4j/java_gateway.py", line 1207, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/pyspark-library/.venv/lib/python3.7/site-packages/pyspark/python/lib/py4j-0.10.9-src.zip/py4j/java_gateway.py", line 1033, in send_command
    response = connection.send_command(command)
  File "/pyspark-library/.venv/lib/python3.7/site-packages/pyspark/python/lib/py4j-0.10.9-src.zip/py4j/java_gateway.py", line 1212, in send_command
    "Error while receiving", e, proto.ERROR_ON_RECEIVE)
py4j.protocol.Py4JNetworkError: Error while receiving


Py4JError: An error occurred while calling o39.read

In [ ]:
def join_acordos(df_bi_recebimento, titulos_vertia):
    
    df_bi_recebimento = (df_bi_recebimento.join(titulos_vertia, 
                           f.ltrim(f.rtrim(df_bi_recebimento['COD_CPFCGC_PES'])) == f.ltrim(f.rtrim(titulos_vertia['COD_CPFCGC_PES'])), "left"))
    return df_bi_recebimento.drop(titulos_vertia['COD_CPFCGC_PES']).drop(titulos_vertia["COD_CONTRATO"])

In [81]:
def join_titulos_vertia(df_bi_recebimento, titulos_vertia):
    df_bi_recebimento = (df_bi_recebimento.join(titulos_vertia, 
                           f.ltrim(f.rtrim(df_bi_recebimento['COD_CPFCGC_PES'])) == f.ltrim(f.rtrim(titulos_vertia['COD_CPFCGC_PES'])), "left"))
    return df_bi_recebimento.drop(titulos_vertia['COD_CPFCGC_PES']).drop(titulos_vertia["COD_CONTRATO"])

In [82]:
def join_originais(df_bi_recebimento, bi_originais):    
    df_bi_recebimento = (df_bi_recebimento.join(bi_originais, 
                           (df_bi_recebimento['COD_PARCELAMENTO'] == bi_originais['COD_PARCELAMENTO_ORI']) &
                           (bi_originais['NOM_NATUREZA_TITULO'] == "MENSALIDADE") &
                           (bi_originais['IND_NATUREZA_REMATRICULA'] == "S"), "left"))
    return df_bi_recebimento.drop(bi_originais['COD_PARCELAMENTO_ORI'])

In [83]:
def join_polos_unidades(df_bi_recebimento, polos_unidades):
    df_bi_recebimento = df_bi_recebimento.drop(bi_carteira_cobranca["COD_FILIAL"])
    df_bi_recebimento = (df_bi_recebimento.join(polos_unidades, 
                           (df_bi_recebimento['COD_FILIAL'] == polos_unidades['COD_CLBR']) &
                           (df_bi_recebimento['COD_FILIAL'] == polos_unidades['COD_OLMP']), "left"))
    
    return df_bi_recebimento.drop(polos_unidades['COD_OLMP']).drop(polos_unidades['COD_CLBR'])

In [ ]:

df.withColumn("gender", when(df.gender == "M","Male") \
      .when(df.gender == "F","Female") \
      .otherwise(df.gender))

In [84]:
## bi contratos tem que ser o primeiro Join
df_bi_recebimento = join_bi_contrato(df_bi_recebimento, bi_contrato)

In [87]:
df_bi_recebimento.printSchema()

root
 |-- COD_REGISTRO: string (nullable = true)
 |-- TIP_MATRICULA: string (nullable = true)
 |-- NOM_MARCA_CONTRATO: string (nullable = true)
 |-- COD_CLIENTE: string (nullable = true)
 |-- NOM_CLIENTE: string (nullable = true)
 |-- COD_CONTRATO: string (nullable = true)
 |-- NUM_CPF_CNPJ: string (nullable = true)
 |-- DSC_SCORE_GLOBAL: string (nullable = true)
 |-- DSC_SCORE_ENGAJAMENTO: string (nullable = true)
 |-- DSC_CLUSTER: string (nullable = true)
 |-- NOM_PRODUTO_PEP: string (nullable = true)
 |-- DSC_IC_FNU: string (nullable = true)
 |-- DSC_IC_FIES: string (nullable = true)
 |-- COD_TITULO: string (nullable = true)
 |-- NUM_NO_TITULO: string (nullable = true)
 |-- TIP_NATUREZA_TITULO: string (nullable = true)
 |-- DSC_NATUREZA_TITULO: string (nullable = true)
 |-- NUM_MES_COMPETENCIA: string (nullable = true)
 |-- NUM_ANO_COMPETENCIA: string (nullable = true)
 |-- VLR_LIQUIDO: string (nullable = true)
 |-- VLR_SEM_DESC_PONTUAL: string (nullable = true)
 |-- DAT_VENCIMENTO:

In [67]:
df_bi_recebimento = join_recebimento(bi_recebimento, bi_acordos_promessas)

In [68]:
df_bi_recebimento = join_carteira_cobranca(df_bi_recebimento, bi_carteira_cobranca)

In [ ]:
df_bi_recebimento = join_titulos_vertia(df_bi_recebimento, titulos_vertia)

In [37]:
df_bi_recebimento = join_originais(df_bi_recebimento, bi_originais)

In [19]:
df_bi_recebimento = join_polos_unidades(df_bi_recebimento, polos_unidades)

In [20]:
# 3)  validacao qtd - OK 
df_bi_recebimento = (df_bi_recebimento
    .drop(bi_contrato["NOM_MARCA_CONTRATO"])
    .drop(bi_originais["STA_TITULO"])
    .drop(titulos_vertia["STA_TITULO"])
    .drop(bi_contrato["TIP_MATRICULA"])
    .drop(titulos_vertia["DAT_PAGAMENTO"])
    )
df_bi_recebimento = (df_bi_recebimento.filter(f.col("STA_TITULO") == "PAGO"))    

In [21]:
# 3) validacao qtd - OK 
df_bi_recebimento.filter(f.col("STA_TITULO") == "PAGO").count()

173376

In [23]:
df_bi_recebimento.printSchema()

root
 |-- COD_REGISTRO: string (nullable = true)
 |-- TIP_MATRICULA: string (nullable = true)
 |-- NOM_MARCA_CONTRATO: string (nullable = true)
 |-- COD_CLIENTE: string (nullable = true)
 |-- NOM_CLIENTE: string (nullable = true)
 |-- COD_CONTRATO: string (nullable = true)
 |-- NUM_CPF_CNPJ: string (nullable = true)
 |-- DSC_SCORE_GLOBAL: string (nullable = true)
 |-- DSC_SCORE_ENGAJAMENTO: string (nullable = true)
 |-- DSC_CLUSTER: string (nullable = true)
 |-- NOM_PRODUTO_PEP: string (nullable = true)
 |-- DSC_IC_FNU: string (nullable = true)
 |-- DSC_IC_FIES: string (nullable = true)
 |-- COD_TITULO: string (nullable = true)
 |-- NUM_NO_TITULO: string (nullable = true)
 |-- TIP_NATUREZA_TITULO: string (nullable = true)
 |-- DSC_NATUREZA_TITULO: string (nullable = true)
 |-- NUM_MES_COMPETENCIA: string (nullable = true)
 |-- NUM_ANO_COMPETENCIA: string (nullable = true)
 |-- VLR_LIQUIDO: string (nullable = true)
 |-- VLR_SEM_DESC_PONTUAL: string (nullable = true)
 |-- DAT_VENCIMENTO:

In [ ]:
 ## OK 
    df_bi_recebimento = (df_bi_recebimento
    .drop(bi_contrato["NOM_MARCA_CONTRATO"])
    .drop(bi_originais["STA_TITULO"])
    .drop(titulos_vertia["STA_TITULO"])
    .drop(bi_contrato["TIP_MATRICULA"])
    .drop(titulos_vertia["DAT_PAGAMENTO"])
    )
df_bi_recebimento = (df_bi_recebimento.filter(f.col("STA_TITULO") == "PAGO"))    

In [ ]:
#### o DATA TYPE Do Campo DAT_PAGAMENTO deve ser ajustado na origem

###   PRECISA MODIFICAR O DATA_TYPE Do campo DAT_PAGAMENTO
df_bi_recebimento = (df_bi_recebimento
                        .withColumn("DSC_CANAL_REPORT",
                        f.when(f.ltrim(f.rtrim(f.col("NOM_USUARIO_ACORDO"))) == "raid-portal-aluno", "raid-portal-aluno")
                        .when((f.col("NOM_GRUPO_USUARIO").isNull()) & (f.ltrim(f.rtrim(f.col("NOM_USUARIO_ACORDO"))) == "adm"), "ATP Proprio/Demais canais")
                        .when(((f.lower(f.col("NOM_USUARIO_ACORDO")).like("%-mal%")) &
                               (f.lower(f.col("NOM_GRUPO_USUARIO")).like("%ATP%MALTA%")) &
                               (f.col("DAT_PAGAMENTO") >= '2021-09-10')))
                        .otherwise(
                            f.col("NOM_GRUPO_USUARIO")
                    )))

In [ ]:
## Ajustei o primeiro Item para "PRESENCIAL"
df_bi_recebimento = (df_bi_recebimento
                        .withColumn("NOM_CARTEIRA",
                        f.when(("TIP_MATRICULA" == f.lit("PRESENCIAL")) & (f.col("NOM_MARCA_CONTRATO").isin('ANHANGUERA','UNIDERP')), "AEDU PRESENCIAL")
                        .when(("TIP_MATRICULA" == f.lit("EAD")) & (f.col("NOM_MARCA_CONTRATO").isin('ANHANGUERA','UNIDERP')), "AEDU EAD")
                        .when(("TIP_MATRICULA" == f.lit("PRESENCIAL")) & (f.col("NOM_MARCA_CONTRATO").isin('FAMA','PITAGORAS','UNIC','UNIME','UNOPAR','UNIRONDON')), "DM PRESENCIAL")
                        .when(("TIP_MATRICULA" == f.lit("EAD")) & (f.col("TIP_MATRICULA").isin('UNIME','UNOPAR','PITAGORAS','UNIASSELVI','PITAGORAS','FAMA-MACAPA','UNIC','UNIRONDON')), "UNOPAR EAD")
                    ))

In [ ]:
## Essa regra nao existe será substituida pelo codigo abaixo 
df_bi_recebimento = (df_bi_recebimento.withColumn("NOM_CANAL_COMISSAO", f.lit("RAID_ATP_INTERVALOR")))

df_bi_recebimento = (df_bi_recebimento.filter(
                      (f.lower(f.col("NOM_USUARIO_ACORDO")).like("%-mal%")) &
                      (f.lower(f.col("NOM_GRUPO_USUARIO")).like("%atp_malta%")) &
                      (f.col("DAT_PAGAMENTO") >= '2021-09-10')
                    ))

In [ ]:
###   substitui o codigo acima
df_bi_recebimento = (df_bi_recebimento
                        .withColumn("NOM_AGENCIA_EXTERNA",
                        f.when(f.ltrim(f.rtrim(f.col("NOM_USUARIO_ACORDO"))) == "raid-portal-aluno", "raid-portal-aluno")
                        .when((f.col("NOM_GRUPO_USUARIO").isNull()) & (f.ltrim(f.rtrim(f.col("NOM_USUARIO_ACORDO"))) == "adm"), "ATP Proprio/Demais canais")
                        .when(((f.lower(f.col("NOM_USUARIO_ACORDO")).like("%-mal%")) &
                               (f.lower(f.col("NOM_GRUPO_USUARIO")).like("%ATP%MALTA%")) &
                               (f.col("DAT_PAGAMENTO") >= '2021-09-10')))
                        .otherwise(
                            f.col("NOM_GRUPO_USUARIO")
                    )))